# Silver Exploration Notebook


In [2]:
import os, findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract, col

spark = SparkSession.builder.appName("silver-exploration").getOrCreate()

RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")

trips = (spark.read
         .option("header", True)
         .csv(f"{RAW}/trips/*/*.csv"))        # all cities, all months

trips.printSchema()
print(trips.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/23 12:44:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/23 12:44:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)



8412973


In [3]:
weather = (spark.read
           .option("multiLine", True)
           .json(f"{RAW}/weather/*/*.json"))

weather.printSchema()

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- apparent_temperature: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- cloud_cover: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- rain: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- snowfall: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- weather_code: array (nullable = true)
 |    |    |-- el

In [4]:
trips.show(5, truncate=False)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------------------+--------------------+---------------------+
|started_at                      |ended_at                        |duration|start_station_id|start_station_name       |start_station_description   |start_station_latitude|start_station_longitude|end_station_id|end_station_name  |end_station_description|end_station_latitude|end_station_longitude|
+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------------------+--------------------+---------------------+
|2022-08-01 03:08:45.036000+00:00|2022-08-01 03:11:38.571000+00:00|173     |564             |Oscars gate     

In [5]:
trips = (trips
         .withColumn("src", input_file_name())
         .withColumn("city", regexp_extract("src", r"/trips/([^/]+)/", 1))
         .withColumn("year_month", regexp_extract("src", r"/([0-9]{4}-[0-9]{2})\.csv", 1)))

trips.groupBy("city").count().show()

+---------+-------+
|     city|  count|
+---------+-------+
|     oslo|5708508|
|   bergen|1901913|
|trondheim| 802552|
+---------+-------+



In [6]:
from pyspark.sql.functions import length
trips.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips.select("started_at", "ended_at").distinct().limit(10).show(truncate=False)

+---+-------+
|len|  count|
+---+-------+
| 25|   8176|
| 32|8404797|
+---+-------+



26/10/23 12:44:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------------------------+--------------------------------+
|started_at                      |ended_at                        |
+--------------------------------+--------------------------------+
|2022-08-01 05:14:13.781000+00:00|2022-08-01 05:26:27.585000+00:00|
|2022-08-01 05:26:55.285000+00:00|2022-08-01 05:38:06.829000+00:00|
|2022-08-01 06:59:11.307000+00:00|2022-08-01 07:03:22.617000+00:00|
|2022-08-01 08:21:36.946000+00:00|2022-08-01 08:28:00.270000+00:00|
|2022-08-01 08:28:31.075000+00:00|2022-08-01 08:34:24.404000+00:00|
|2022-08-01 09:47:55.571000+00:00|2022-08-01 09:49:27.312000+00:00|
|2022-08-01 12:34:58.312000+00:00|2022-08-01 12:46:18.628000+00:00|
|2022-08-01 14:18:11.927000+00:00|2022-08-01 14:34:37.844000+00:00|
|2022-08-01 14:44:44.092000+00:00|2022-08-01 14:55:35.589000+00:00|
|2022-08-01 14:47:03.953000+00:00|2022-08-01 14:48:43.623000+00:00|
+--------------------------------+--------------------------------+



##### Checking time stamps

In [7]:
from pyspark.sql.functions import min, max, sum as spark_sum, when
t = trips.withColumn("duration_s", col("duration").cast("double"))
t.select(min("duration_s"), max("duration_s")).show()
t.select(spark_sum(when(col("duration_s") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration_s") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration_s").isNull(), 1).otherwise(0)).alias("null_duration")).show()

+---------------+---------------+
|min(duration_s)|max(duration_s)|
+---------------+---------------+
|            0.0|        79790.0|
+---------------+---------------+



+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1260|            0|
+------------+---------+-------------+



In [8]:
from pyspark.sql.functions import arrays_zip, explode

w = (weather
     .withColumn("src", input_file_name())
     .select("src", explode(arrays_zip(
         col("hourly.time"),
         col("hourly.temperature_2m"),
         col("hourly.rain"),
         col("hourly.wind_speed_10m"),
        col("hourly.apparent_temperature"),
        col("hourly.precipitation")
     )).alias("h")))

w.select("src", "h.*").show(3, truncate = False)

+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|src                                                                                                                    |time            |temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T00:00|0.5           |0.0 |7.7           |-3.0                |0.0          |
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T01:00|0.2           |0.0 |7.6   

In [9]:
trips.filter(length("started_at") == 25).select("started_at", "ended_at").show(5, truncate=False)

from pyspark.sql.functions import to_timestamp, length

p = (trips
     .withColumn("start_ts", to_timestamp("started_at"))
     .withColumn("end_ts", to_timestamp("ended_at"))
     .withColumn("len_start", length("started_at"))
     .withColumn("len_end", length("ended_at")))

p.groupBy("len_start", "len_end").count().show()
p.filter(col("start_ts").isNull() | col("end_ts").isNull()).count()

from pyspark.sql.functions import regexp_extract, input_file_name, length, count, sum as spark_sum, when

chk = (trips
       .withColumn("src", input_file_name())
       .withColumn("city", regexp_extract("src", r"/trips/([^/]+)/", 1))
       .withColumn("year_month", regexp_extract("src", r"/([0-9]{4}-[0-9]{2})\.csv", 1))
       .groupBy("city", "year_month")
       .agg(count("*").alias("rows"),
            spark_sum(when(length("started_at") == 25, 1).otherwise(0)).alias("short_start")))

chk.withColumn("pct", col("short_start") / col("rows") * 100) \
   .orderBy(col("pct").desc()).show(10)

+-------------------------+--------------------------------+
|started_at               |ended_at                        |
+-------------------------+--------------------------------+
|2022-08-01 13:05:31+00:00|2022-08-01 13:18:14.855000+00:00|
|2022-08-01 14:54:32+00:00|2022-08-01 15:01:39.232000+00:00|
|2022-08-01 16:01:52+00:00|2022-08-01 16:24:06.468000+00:00|
|2022-08-01 16:30:21+00:00|2022-08-01 16:37:31.473000+00:00|
|2022-08-01 17:07:22+00:00|2022-08-01 17:25:54.295000+00:00|
+-------------------------+--------------------------------+
only showing top 5 rows



+---------+-------+-------+
|len_start|len_end|  count|
+---------+-------+-------+
|       32|     32|8396750|
|       32|     25|   8047|
|       25|     32|   8171|
|       25|     25|      5|
+---------+-------+-------+



+---------+----------+-----+-----------+-------------------+
|     city|year_month| rows|short_start|                pct|
+---------+----------+-----+-----------+-------------------+
|trondheim|   2024-06|21822|         35|0.16038859866190083|
|   bergen|   2025-07|28903|         43|  0.148773483721413|
|trondheim|   2026-10|19088|         27| 0.1414501257334451|
|     oslo|   2025-12|24317|         34|0.13981987909692808|
|   bergen|   2023-10|42989|         60| 0.1395705878247924|
|trondheim|   2024-07|17209|         24| 0.1394619094659771|
|trondheim|   2023-05|16792|         23|0.13696998570747976|
|trondheim|   2026-03| 2952|          4|0.13550135501355012|
|   bergen|   2026-02|11837|         16|0.13516938413449353|
|   bergen|   2023-03|30451|         41|0.13464254047486124|
+---------+----------+-----+-----------+-------------------+
only showing top 10 rows



In [10]:
t = trips.withColumn("duration_s", col("duration").cast("double"))
print(t.approxQuantile("duration_s", [0.5, 0.99, 0.999, 0.9999], 0.001))
print(t.filter(col("duration_s") > 6*3600).count())
print(t.filter(col("duration_s") > 12*3600).count())
print(t.approxQuantile("duration_s", [0.999, 0.9999], 0.0001))

[524.0, 3706.0, 79790.0, 79790.0]


315


15


[9142.0, 79790.0]


In [11]:
from pyspark.sql.functions import to_timestamp

p = (trips
     .withColumn("start_ts", to_timestamp("started_at"))
     .withColumn("end_ts", to_timestamp("ended_at")))

print(p.filter(col("start_ts").isNull() | col("end_ts").isNull()).count())

0


The check above confirms that every one of hte 8.4 million trips has a start time and an end time that Spark read.

In [12]:
from pyspark.sql.functions import unix_timestamp, abs as spark_abs

d = (p
     .withColumn("duration_s", col("duration").cast("double"))
     .withColumn("diff_s", unix_timestamp("end_ts") - unix_timestamp("start_ts")))

print(d.filter(spark_abs(col("diff_s") - col("duration_s")) > 2).count())

0


Counts the rows where the gap between start and end differs from the stated duration by more than 2 seconds.

##### Check if any station fields are empty

Make sure each trip has a starting station

In [14]:
from pyspark.sql.functions import sum as spark_sum, when

cols = ["start_station_id", "end_station_id",
        "start_station_latitude", "start_station_longitude",
        "end_station_latitude", "end_station_longitude"]

trips.select([
    spark_sum(when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)).alias(c)
    for c in cols
]).show()

+----------------+--------------+----------------------+-----------------------+--------------------+---------------------+
|start_station_id|end_station_id|start_station_latitude|start_station_longitude|end_station_latitude|end_station_longitude|
+----------------+--------------+----------------------+-----------------------+--------------------+---------------------+
|               0|             0|                     0|                      0|                   0|                    0|
+----------------+--------------+----------------------+-----------------------+--------------------+---------------------+



##### Check if any trips are duplicated


In [15]:
total = trips.count()
distinct = trips.dropDuplicates().count()
print(total, distinct, total - distinct)

26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/10/23 12:48:52 WARN RowBasedKeyValueBatch: Calling spill() on

8412973 8412948 25


### Weather Analysis

In [16]:
from pyspark.sql.functions import regexp_extract

wx = (w.select("src", "h.*")
      .withColumn("city", regexp_extract("src", r"/weather/([^/]+)/", 1)))

print(wx.count())
print(wx.select("city", "time").distinct().count())

127152
126216


First number is the total number of weather rows, and the second is the number of distinct (city, hour) pairs. Need to come up with a rule that deals with duplicates that get re-fetched.

In [17]:
from pyspark.sql.functions import count, countDistinct

dups = (wx.groupBy("city", "time")
        .agg(count("*").alias("n"),
             countDistinct("temperature_2m").alias("temps"),
             countDistinct("precipitation").alias("precs"))
        .filter(col("n") > 1))

print(dups.count())
print(dups.filter((col("temps") > 1) | (col("precs") > 1)).count())

936


0


##### Check if any weather values are missing

In [19]:
from pyspark.sql.functions import sum as spark_sum, when

weather_cols = ["temperature_2m", "rain", "wind_speed_10m",
                "apparent_temperature", "precipitation"]

wx.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in weather_cols
]).show()

+--------------+----+--------------+--------------------+-------------+
|temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+--------------+----+--------------+--------------------+-------------+
|             0|   0|             0|                   0|            0|
+--------------+----+--------------+--------------------+-------------+

